In [ ]:
!pip install faster-whisper numpy

  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 51.6 MB/s eta 0:00:00


In [4]:
import subprocess
import sys
print("Setting up environment and installing packages... please wait")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=True)

import os
import base64
import io
import torch
import numpy as np
from faster_whisper import WhisperModel
from IPython.display import HTML, display
from google.colab import output

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "float32"

print(f"Loading [MEDIUM] model on {device.upper()}...")
model = WhisperModel("medium", device=device, compute_type=compute_type)
print("Medium model loaded successfully and ready for fast streaming!")

audio_buffer = np.zeros(16000 * 3, dtype=np.float32)

def process_audio_chunk(base64_data):
    global audio_buffer
    if not base64_data:
        return ""

    audio_bytes = base64.b64decode(base64_data)
    chunk = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0

    if len(chunk) == 0:
        return ""

    audio_buffer = np.roll(audio_buffer, -len(chunk))
    audio_buffer[-len(chunk):] = chunk

    segments, _ = model.transcribe(
        audio_buffer,
        beam_size=1,
        language="ar",
        no_speech_threshold=0.6,
        log_prob_threshold=-1.0
    )

    text_output = "".join([segment.text for segment in segments])
    return text_output.strip()

output.register_callback('notebook.process_audio_chunk', process_audio_chunk)

AUDIO_HTML = """
<div style="padding: 20px; background: #111; color: #fff; border-radius: 12px; font-family: sans-serif; max-width: 500px; box-shadow: 0 4px 15px rgba(0,0,0,0.5);">
    <h3 style="color: #FF9800; margin-top: 0;"> Whisper Medium Ultra-Fast Streaming</h3>
    <p style="font-size: 13px; color: #aaa;">Streaming is accelerated to update roughly every half second:</p>
    <button id="recordBtn" onclick="toggleRecording()" style="padding: 12px 24px; background: #FF9800; color: white; border: none; border-radius: 6px; font-size: 15px; cursor: pointer; font-weight: bold; width: 100%; transition: 0.3s;">🔴 Start Speaking Now</button>
    <div id="status" style="margin-top: 15px; color: #ffeb3b; font-weight: bold; text-align: center;"> Ready</div>
    <div style="margin-top: 15px; font-size: 12px; color: #888;">Detected text (high-speed instant updates):</div>
    <div id="transcriptBox" style="margin-top: 5px; padding: 15px; background: #222; border: 1px solid #333; border-radius: 6px; min-height: 60px; font-size: 16px; color: #00ffcc; direction: rtl; line-height: 1.5;">...</div>
</div>

<script>
let audioContext;
let processor;
let globalStream;
let isRecording = false;

async function toggleRecording() {
    const btn = document.getElementById('recordBtn');
    const status = document.getElementById('status');

    if (!isRecording) {
        try {
            globalStream = await navigator.mediaDevices.getUserMedia({ audio: true });
            audioContext = new (window.AudioContext || window.webkitAudioContext)({ sampleRate: 16000 });
            const input = audioContext.createMediaStreamSource(globalStream);

            // Lowered to 8192 to force the browser to send audio batches every half second for ultra-fast streaming
            processor = audioContext.createScriptProcessor(8192, 1, 1);
            input.connect(processor);
            processor.connect(audioContext.destination);

            processor.onaudioprocess = async (e) => {
                if (!isRecording) return;
                const leftChannel = e.inputBuffer.getChannelData(0);
                const lInt16 = new Int16Array(leftChannel.length);
                for (let i = 0; i < leftChannel.length; i++) {
                    lInt16[i] = leftChannel[i] * 32767;
                }

                const base64Audio = btoa(String.fromCharCode.apply(null, new Uint8Array(lInt16.buffer)));

                const result = await google.colab.kernel.invokeFunction('notebook.process_audio_chunk', [base64Audio], {});
                if (result && result.data && result.data['text/plain']) {
                    let text = result.data['text/plain'].replace(/^['"]|['"]$/g, '');
                    if(text.trim() != "") {
                        document.getElementById('transcriptBox').innerText = text;
                    }
                }
            };

            isRecording = true;
            btn.innerText = "Stop Streaming";
            btn.style.background = "#f44336";
            status.innerText = " Continuous live streaming... keep talking without stopping:";

        } catch (err) {
            alert("Failed to access microphone: " + err);
        }
    } else {
        isRecording = false;
        if (processor) processor.disconnect();
        if (globalStream) globalStream.getTracks().forEach(track => track.stop());
        if (audioContext) audioContext.close();

        btn.innerText = "Start Speaking Now";
        btn.style.background = "#FF9800";
        status.innerText = " Stopped.";
    }
}
</script>
"""

display(HTML(AUDIO_HTML))

Setting up environment and installing packages... please wait
Loading [MEDIUM] model on CUDA...
Medium model loaded successfully and ready for fast streaming!
